# 04 — Rank feature lab

Sandbox for FeatureVector design and ranking experiments. Scoring is the
production algorithm: StandardScaler + max cosine to Seeds (`rank_dayhikes` /
`rank_scaled_rows`). **Do not train a ranker here.**

Production ranking itself is walked through in [`03_ranking.ipynb`](03_ranking.ipynb).
Load GeometryStore as GeoPandas in [`04a_geometry.ipynb`](04a_geometry.ipynb).
Geometry and raw amenity points live in GeometryStore
(`{catalog}.geometry.sqlite`), not in Catalog. Production AmenityFacts stay
Trailhead-disk counts of hut/parking/restaurant_cafe/drinking_water.

## How to use this as a sandbox

1. Pick a Seed (default: first OSM-named DayHike).
2. Look at which production columns actually vary.
3. Add numeric columns in **Experimental columns** (GeometryStore, DEM, anything).
4. Run `rank_with_columns` and read the comparison table.
5. Ablate noisy columns.

Nothing here writes back to the Catalog. Promoting a column means changing
`FeatureVector` / ingest in the package.

Open `data/catalog.sqlite` plus `data/catalog.geometry.sqlite` when both exist.
Otherwise build the committed fixture.

DEM / SolarAspect: `dayhike fetch-dem` writes FABDEM GeoTIFFs into `data/dem/`
(gitignored). `dayhike sample-dem` copies those heights onto GeometryStore
vertices and Catalog elevation. This notebook samples rasters for SolarAspect;
without a covering file it prints `dem skipped` and still ranks.


In [ ]:
import json
import math
import os
import sqlite3
import tempfile
from pathlib import Path

repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

import numpy as np
from sklearn.neighbors import BallTree

from dayhike import (
    Catalog,
    build_catalog,
    feature_matrix,
    fit_scaler_from_matrix,
    geometry_store_path_for,
    rank_dayhikes,
    rank_scaled_rows,
    scale_matrix,
)
from dayhike.catalog import EARTH_RADIUS_M, FEATURE_NAMES

local = repo / "data" / "catalog.sqlite"
local_geometry = geometry_store_path_for(local)
if (
    local.is_file()
    and local_geometry.is_file()
    and "PYTEST_CURRENT_TEST" not in os.environ
):
    catalog_path = local
    geometry_path = local_geometry
    catalog = Catalog.open(catalog_path)
else:
    catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    catalog = build_catalog(repo / "tests" / "fixtures" / "catalog.osm", catalog_path)
    geometry_path = geometry_store_path_for(catalog_path)

geo = sqlite3.connect(geometry_path)
print("catalog", catalog_path)
print("geometry_store", geometry_path)
print(catalog.ingest_report)
print("name_origin")
for origin, n in geo.execute(
    "SELECT name_origin, COUNT(*) FROM dayhikes GROUP BY 1 ORDER BY 1"
):
    print(f"  {origin}\t{n}")


## Production baseline

`rank_dayhikes` is the control. Filter `name_origin != 'synthetic'` in
postprocessing if you only want OSM names. Change `SEED_NAME` below to try
another hike from your Catalog.


In [ ]:
SEED_NAME = next(
    hike.name for hike in catalog.dayhikes if hike.name_origin == "osm_name"
)
print("seed", SEED_NAME)
production = rank_dayhikes(catalog, [SEED_NAME])
for item in production.items:
    print(
        f"prod_rank\t{item.dayhike.id}\t{item.dayhike.name}\t"
        f"{item.dayhike.name_origin}\t{item.score:.6f}"
    )

named = [hike for hike in catalog.dayhikes if hike.name_origin == "osm_name"]
print("osm_name_only", len(named), "of", len(catalog.dayhikes))


## What the production columns do

Missingness and variance tell you which Catalog columns can move a ranking.
A column that is all-NaN or constant cannot separate neighbours.


In [ ]:
import os

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")

import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)
import pandas as pd

ids = [hike.id for hike in catalog.dayhikes]
names = [hike.name for hike in catalog.dayhikes]
by_id = {hike.id: hike for hike in catalog.dayhikes}
base = feature_matrix(catalog.dayhikes)
prod_frame = pd.DataFrame(base, columns=FEATURE_NAMES, index=ids)
summary = pd.DataFrame(
    {
        "missing": prod_frame.isna().mean(),
        "std": prod_frame.std(numeric_only=True),
        "min": prod_frame.min(numeric_only=True),
        "max": prod_frame.max(numeric_only=True),
    }
)
print(summary.to_string())

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
plot_cols = [
    "length_m",
    "elevation_gain_m",
    "difficulty_ordinal",
    "hut_count",
    "parking_nearest_m",
    "drinking_water_count",
]
for ax, col in zip(axes.ravel(), plot_cols):
    values = prod_frame[col].dropna()
    if values.empty:
        ax.set_title(f"{col} (all missing)")
        continue
    ax.hist(values, bins=min(20, max(5, values.nunique())), color="#0f766e", edgecolor="white")
    ax.set_title(col)
fig.tight_layout()
show_fig(fig)


## Sandbox helpers

`rank_with_columns` is the whole experimental loop: stack a matrix, fit a
scaler, max-cosine rank. `compare_to_production` prints rank/score deltas.
Edit **Experimental columns** in the next code cell; leave a column out of
`EXP_NAMES` to ablate it.


In [ ]:
def rank_with_columns(matrix, feature_names, seed_ids):
    scaler = fit_scaler_from_matrix(matrix, feature_names)
    return rank_scaled_rows(ids, names, scale_matrix(matrix, scaler), seed_ids)


def compare_to_production(experimental, label="experiment"):
    prod_rank = {item.dayhike.id: i for i, item in enumerate(production.items, start=1)}
    prod_score = {item.dayhike.id: item.score for item in production.items}
    rows = []
    for rank, (dayhike_id, name, score) in enumerate(experimental, start=1):
        rows.append(
            {
                "id": dayhike_id,
                "name": name,
                "prod_rank": prod_rank.get(dayhike_id),
                f"{label}_rank": rank,
                "delta_rank": (prod_rank.get(dayhike_id) or 0) - rank,
                "prod_score": prod_score.get(dayhike_id),
                f"{label}_score": score,
            }
        )
    table = pd.DataFrame(rows)
    print(table.head(15).to_string(index=False))
    order_changed = [row[0] for row in experimental] != [
        item.dayhike.id for item in production.items
    ]
    print("order_changed", order_changed)
    return table


## Experimental columns

Default extras from GeometryStore (not Catalog AmenityFacts):

- `vertex_count` — how many points the assembled geometry has.
- `sinuosity` — route length / start–end great-circle (1 = straight).
- `corridor_hut_count` — huts within 500 m of **DayHikeGeometry**, not the
  Trailhead disk.

Add your own arrays of `len(ids)` and append the name to `EXP_NAMES`.


In [ ]:
vertex_count_by_id = dict(
    geo.execute("SELECT dayhike_id, COUNT(*) FROM vertices GROUP BY 1")
)
vertex_count = np.array(
    [float(vertex_count_by_id.get(hike_id, 0)) for hike_id in ids], dtype=float
)

first_last = {
    dayhike_id: (lat1, lon1, lat2, lon2)
    for dayhike_id, lat1, lon1, lat2, lon2 in geo.execute(
        '''
        SELECT a.dayhike_id, a.latitude, a.longitude, b.latitude, b.longitude
        FROM vertices a
        JOIN vertices b ON b.dayhike_id = a.dayhike_id
        WHERE a.seq = 0
          AND b.seq = (SELECT MAX(seq) FROM vertices v WHERE v.dayhike_id = a.dayhike_id)
        '''
    )
}


def haversine_m(lat1, lon1, lat2, lon2):
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * EARTH_RADIUS_M * math.asin(math.sqrt(a))


sinuosity = np.ones(len(ids), dtype=float)
for index, hike_id in enumerate(ids):
    ends = first_last.get(hike_id)
    if not ends:
        continue
    chord = haversine_m(*ends)
    if chord > 0:
        sinuosity[index] = by_id[hike_id].length_m / chord

hut_points = [
    (lat, lon)
    for lat, lon in geo.execute(
        "SELECT latitude, longitude FROM amenities WHERE kind = 'hut'"
    )
]
corridor_huts = np.zeros(len(ids), dtype=float)
if hut_points:
    tree = BallTree(np.radians(np.asarray(hut_points)), metric="haversine")
    radius = 500.0 / EARTH_RADIUS_M
    verts_by_id: dict[int, list] = {}
    for hike_id, lat, lon in geo.execute(
        "SELECT dayhike_id, latitude, longitude FROM vertices ORDER BY dayhike_id, seq"
    ):
        verts_by_id.setdefault(hike_id, []).append((lat, lon))
    for index, hike_id in enumerate(ids):
        verts = verts_by_id.get(hike_id) or []
        if not verts:
            continue
        hits = tree.query_radius(np.radians(np.asarray(verts)), r=radius)
        found: set[int] = set()
        for neighbours in hits:
            found.update(int(i) for i in neighbours)
        corridor_huts[index] = float(len(found))

# --- edit here ---
extra = {
    "vertex_count": vertex_count,
    "sinuosity": sinuosity,
    "corridor_hut_count": corridor_huts,
}
EXP_NAMES = FEATURE_NAMES + tuple(extra)
exp_matrix = np.column_stack([base, *[extra[name] for name in extra]])
seed_hike = by_id[next(hike.id for hike in catalog.dayhikes if hike.name == SEED_NAME)]
exp_ranking = rank_with_columns(exp_matrix, EXP_NAMES, {seed_hike.id})
for dayhike_id, name, score in exp_ranking:
    print(f"exp_rank\t{dayhike_id}\t{name}\t{score:.6f}")
compare_to_production(exp_ranking)


## Ablation

Drop Catalog columns (here: elevation) and see whether order moves. Use the
same helper with a column mask when you want to test "does this feature earn
its keep?".


In [ ]:
keep = [i for i, name in enumerate(FEATURE_NAMES) if not name.startswith("elevation_")]
ablate_names = tuple(FEATURE_NAMES[i] for i in keep)
ablate_matrix = base[:, keep]
ablate_ranking = rank_with_columns(ablate_matrix, ablate_names, {seed_hike.id})
print("dropped", [name for name in FEATURE_NAMES if name.startswith("elevation_")])
compare_to_production(ablate_ranking, label="no_elevation")


## Map the Seed and its neighbours

GeometryStore vertices for the Seed plus the top production neighbours.
Amenities are the raw points (including Viewpoint / Peak / Shelter).


In [ ]:
from shapely.geometry import LineString, Point
import geopandas

show_ids = {seed_hike.id, *[item.dayhike.id for item in production.items[:8]]}
vertex_rows = list(
    geo.execute(
        "SELECT dayhike_id, seq, latitude, longitude FROM vertices "
        f"WHERE dayhike_id IN ({','.join('?' * len(show_ids))}) ORDER BY dayhike_id, seq",
        tuple(show_ids),
    )
)
by_hike = {}
for dayhike_id, _seq, lat, lon in vertex_rows:
    by_hike.setdefault(dayhike_id, []).append((lon, lat))
lines = [
    {
        "id": dayhike_id,
        "name": by_id[dayhike_id].name,
        "is_seed": dayhike_id == seed_hike.id,
        "geometry": LineString(coords),
    }
    for dayhike_id, coords in by_hike.items()
    if len(coords) >= 2
]
gdf = geopandas.GeoDataFrame(lines, crs="EPSG:4326")
amenity_gdf = geopandas.GeoDataFrame(
    [
        {"kind": kind, "geometry": Point(lon, lat)}
        for kind, lat, lon in geo.execute(
            "SELECT kind, latitude, longitude FROM amenities"
        )
    ],
    crs="EPSG:4326",
)
fig, ax = plt.subplots(figsize=(6, 6))
if not gdf.empty:
    neighbours = gdf.loc[~gdf.is_seed]
    seed_gdf = gdf.loc[gdf.is_seed]
    if not neighbours.empty:
        neighbours.plot(ax=ax, color="#94a3b8", linewidth=1.5)
    if not seed_gdf.empty:
        seed_gdf.plot(ax=ax, color="#c2410c", linewidth=3)
if not amenity_gdf.empty:
    amenity_gdf.plot(ax=ax, column="kind", markersize=20, legend=True)
ax.set_title(f"Seed {SEED_NAME!r} + neighbours")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
fig.tight_layout()
show_fig(fig)
print("mapped_dayhikes", len(gdf), "amenities", len(amenity_gdf))


## Elevation profile (GeometryStore `vertices.ele`)

OSM `ele` on the fixture Ridge Walk; FABDEM metres after `sample-dem` on a
real Catalog. This is not SolarAspect — it is the along-track height the
Catalog elevation stats come from.


In [ ]:
profile = list(
    geo.execute(
        "SELECT seq, latitude, longitude, ele FROM vertices WHERE dayhike_id = ? ORDER BY seq",
        (seed_hike.id,),
    )
)
seq = [row[0] for row in profile]
ele = [row[3] for row in profile]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(seq, ele, color="#0f766e", marker="o")
ax.set_xlabel("vertex seq")
ax.set_ylabel("ele (m)")
ax.set_title(f"Elevation profile: {SEED_NAME}")
fig.tight_layout()
show_fig(fig)
print("profile_points", len(profile), "with_ele", sum(z is not None for z in ele))


## DEM and SolarAspect

`dayhike fetch-dem` then `dayhike sample-dem` persist FABDEM heights onto
GeometryStore vertices and Catalog elevation min/max/gain. Sampling here is
for a SolarAspect prototype: finite-difference slope aspect at subsampled
vertices, then mean south-facing-ness (`1` = south, `0` = north).

SolarAspect is **not** a Catalog FeatureVector column until this sandbox
promotes it. Without a covering GeoTIFF this cell prints `dem skipped`.


In [ ]:
from dayhike.dem import DEFAULT_DEM_DIR, sample_elevations

points = [
    (lat, lon)
    for _id, lat, lon in geo.execute(
        "SELECT dayhike_id, latitude, longitude FROM vertices LIMIT 20"
    )
]
samples = sample_elevations(repo / DEFAULT_DEM_DIR, points)
finite = [z for z in samples if z is not None]
if not finite:
    print("dem skipped: no covering FABDEM tile in data/dem/")
else:
    print("dem sampled", "n", len(finite), "min", min(finite), "max", max(finite))
    print("solar aspect is DEM-derived; not live weather")

    def southness_for_hike(hike_id: int) -> float:
        verts = list(
            geo.execute(
                "SELECT latitude, longitude FROM vertices WHERE dayhike_id = ? ORDER BY seq",
                (hike_id,),
            )
        )
        if len(verts) < 2:
            return float("nan")
        step = max(1, len(verts) // 12)
        chosen = verts[::step][:12]
        delta_deg = 30.0 / EARTH_RADIUS_M * (180.0 / math.pi)
        probe = []
        for lat, lon in chosen:
            probe.extend(
                [
                    (lat, lon),
                    (lat, lon + delta_deg / max(math.cos(math.radians(lat)), 0.2)),
                    (lat + delta_deg, lon),
                ]
            )
        z = sample_elevations(repo / DEFAULT_DEM_DIR, probe)
        scores = []
        for i in range(len(chosen)):
            z0, ze, zn = z[3 * i], z[3 * i + 1], z[3 * i + 2]
            if None in (z0, ze, zn):
                continue
            dz_dx = ze - z0
            dz_dy = zn - z0
            aspect = math.atan2(-dz_dy, -dz_dx)
            scores.append((1.0 + math.cos(aspect - math.pi)) / 2.0)
        if not scores:
            return float("nan")
        return float(sum(scores) / len(scores))

    solar = np.array([southness_for_hike(hike_id) for hike_id in ids], dtype=float)
    print("solar_southness", "finite", int(np.isfinite(solar).sum()), "mean", np.nanmean(solar))
    solar_matrix = np.column_stack([base, solar])
    solar_ranking = rank_with_columns(
        solar_matrix, FEATURE_NAMES + ("solar_southness",), {seed_hike.id}
    )
    compare_to_production(solar_ranking, label="solar")
geo.close()


## Promoting a column

When an experimental column earns its keep:

1. Compute it at ingest (or `sample-dem`) so it is stored on Catalog rows.
2. Add it to `FeatureVector` / `FEATURE_NAMES` and the SQLite `feature_vectors`
   table.
3. Refit `scaler` with the rest of the Catalog.
4. Keep DriveTime out of the vector — it stays a query-time filter.

This notebook does not do those writes. Re-run `rank_dayhikes` after the
package change; that becomes the new baseline cell at the top.
